# search-compare — Exemplo de uso

Este notebook demonstra como usar a biblioteca `search_compare` para:

1. Construir duas strings de busca PICOC
2. Executar as buscas na Scopus
3. Comparar os resultados e verificar a cobertura de artigos de controle

## Pré-requisitos

* Obtenha uma chave de API da scopus, o que pode ser feito em:
https://dev.elsevier.com/apikey/manage

    Passe a chave diretamente ao instanciar `ScopusClient`.

    Ou adicione uma variável de ambiente `SCOPUS_API_KEY`.

* E instale o pacote search-compare: ```pip install search-compare```

## 1. Imports

In [ ]:
# Para instalar o pacote:
%pip install search-compare

# Descomentar em Desenvolvimento para utilizar a versão local do pacote
# sys.path.insert(0, "../src")

In [ ]:
import sys

from search_compare import (
    ControlArticle,
    PicocString,
    SearchManager,
    ScopusClient,
)

## 2. Cliente

In [ ]:
client = ScopusClient()  # lê SCOPUS_API_KEY do .env ou variável de ambiente
# client = ScopusClient(api_key="sua-chave")  # ou passe diretamente

## 3. Construir as queries PICOC e executar as buscas

Os filtros comuns são definidos uma vez e reutilizados com `**`.

Parâmetros disponíveis em `build_query`:
- `field`: `"TITLE-ABS-KEY"` (padrão), `"TITLE"`, `"ABS"`, `"KEY"`
- `year_min` / `year_max`: intervalo de publicação
- `language`: ex. `"english"`
- `doc_type`: `"ar"` (artigo), `"cp"` (conference paper), `"re"` (review), `"cr"` (conference review) — aceita lista

In [ ]:
base_filters = {
    "field": "TITLE-ABS-KEY",
    "year_min": 2020,
    "year_max": 2026,
    "language": "english",
}

query1 = client.build_query(PicocString(
    population=['embedded system', 'embedded software', 'embedded application', 'cyber-physical system', 'firmware', 'iot', 'internet of things'],
    intervention=['architectur*', 'design pattern'],
    outcome=['maintainab*', 'testab*'],
    context=['IoT', 'internet of things', 'Cyber-Physical Systems'],
), **base_filters)
result1 = query1.search()

query2 = client.build_query(PicocString(
    population=['software', 'firmware'],
    intervention=['architectur*', 'design pattern'],
    outcome=['maintainab*', 'testab*', 'SOLID'],
    context=['MCU', 'microcontroller', 'embedded', 'cyber-physical system', 'firmware', 'iot', 'internet of things'],
), **base_filters)
result2 = query2.search()

In [32]:
print(f"Query 1:")
print(query1)
print(f"Recuperados: {len(result1)} / Total disponível: {result1.total_count}\n")

print(f"Query 2:")
print(query2)
print(f"Recuperados: {len(result2)} / Total disponível: {result2.total_count}\n")

Query 1:
TITLE-ABS-KEY(("embedded system" OR "embedded software" OR "embedded application" OR "cyber-physical system" OR firmware OR iot OR "internet of things") AND (architectur* OR "design pattern") AND (maintainab* OR testab*) AND (IoT OR "internet of things" OR "Cyber-Physical Systems")) AND PUBYEAR > 2020 AND PUBYEAR < 2026 AND LANGUAGE(english)
Recuperados: 128 / Total disponível: 128

Query 2:
TITLE-ABS-KEY((software OR firmware) AND (architectur* OR "design pattern") AND (maintainab* OR testab* OR SOLID) AND (embedded OR "cyber-physical system" OR firmware OR iot OR "internet of things")) AND PUBYEAR > 2020 AND PUBYEAR < 2026 AND LANGUAGE(english)
Recuperados: 158 / Total disponível: 158



## 4. Definir artigos de controle

Artigos que sabemos que deveriam aparecer na busca.

Cada artigo de controle pode ser identificado por título, ano e Elsevier ID (eid).

Cada parâmetro informado será verificado exatamente.

In [33]:
manager = SearchManager(client=client)

manager.set_control_articles([
    ControlArticle(title="Implementation of a universal framework using design patterns for application development on microcontrollers", year=2024),
    ControlArticle(title="Benefits of Using Design Patterns on Microcontrollers in Implemented IoT Applications", year=2024),
    ControlArticle(title="Component-based Control Software design using IEC 61499 Adapter Interfaces", year=2025),
    ControlArticle(title="Example of non-existent title", year=1970),
])

check = manager.check_control_articles_exists()



✗ Artigo não encontrado na base: 'Example of non-existent title' (1970)
Verifique os dados inseridos e a existência do artigo na base de dados.


## 5. Comparar os resultados

Exibe primeiro os artigos de controle e sua presença em cada query,
depois os documentos em comum e as diferenças.

In [34]:
manager.compare(result1, result2)

### Artigos de controle (3/4 encontrados na base)

year,title,query 1,query 2
2025,Component-based Control Software design using IEC 61499 Adapter Interfaces,✓,✓
2024,Implementation of a universal framework using design patterns for application development on microcontrollers,✓,✓
2024,Benefits of Using Design Patterns on Microcontrollers in Implemented IoT Applications,✗,✓
1970,Example of non-existent title,Não encontrado na base,Não encontrado na base


### Documentos em comum (44)

year,title,authors,source
2025,ISARA: An Island-Style Systolic Array Reconfigurable Accelerator Based on Memristors for Deep Neural Networks,Yang F.,IEEE Transactions on Very Large Scale Integration VLSI Systems
2025,Making Use of Design Patterns in IoT Middleware Implementation,Harjumaa L.,International Conference on Internet of Things Big Data and Security Iotbds Proceedings
2025,Integrating Predictive Analytics and Internet of Things (IoT) to Optimize Supply Chains,Rajendran P.,3rd International Conference on Integrated Circuits and Communication Systems Icicacs 2025
2025,Enhanced Layered Architectures and Semi-formal Representations in Refactoring Using Domain-Driven Design,Uehara S.,Communications in Computer and Information Science
2025,Component-based Control Software design using IEC 61499 Adapter Interfaces,Sharma S.,IEEE International Conference on Emerging Technologies and Factory Automation ETFA
2025,An Intelligent Gamified Reviewer System for Enhancing Student Engagement and Performance in Cyber-Physical Learning Environments,Saballa-Marin M.J.M.,International Conference on Communication Computing Networking and Control in Cyber Physical Systems Ccncps 2025
2025,A Platform-Agnostic Publish–Subscribe Architecture with Dynamic Optimization,Twabi A.,Future Internet
2025,Towards the Integration of Lightweight Components in Cyber-Physical Systems,Mena M.,SN Computer Science
2025,Flexible and Reproducible RF Calibration using Google Cloud Workflows,Keles Y.E.,International Conference on Computer Science and Engineering Ubmk
2025,LLM-Driven Approach to Automated Sustainability of IoT Systems,Petrovic N.,2025 IEEE 34th International Conference on Microelectronics Miel 2025 Proceedings


### Apenas na query 1 (84)

year,title,authors,source
2025,"SSA: An Effective Secure Scan Architecture Based on Hidden, Randomly Inserted Keys and PUF",Wang W.,IEEE Internet of Things Journal
2025,Design and Performance Comparison of Current References for a 32-bit Microcontroller Using 22nm Technology,Barcelos E.A.,2025 IEEE 16th Latin American Symposium on Circuits and Systems Lascas 2025 Proceedings
2025,Design of Metaverse Digital Technology Optimization Model Based on Artificial Intelligence Technology,Li S.,Procedia Computer Science
2025,Three-Dimensional Printed MXene@PANI Hierarchical Architecture for High-Performance Micro-Supercapacitors,Zhang A.,Materials
2025,An integrated blockchain architecturefor sustainable higher education administration,Mishra R.A.,On the Horizon
2025,Higher Layer Design for 3GPP Ambient Internet of Things,Van D.P.,IEEE Communications Standards Magazine
2025,Identifying and Architecting Microservices for Edge Computing,Vora U.,Proceedings 2025 IEEE 22nd International Conference on Software Architecture Icsa C 2025
2025,Interoperability Assessment Oriented Towards the Development of Digital Twin Architecture in Industrial Maintenance,Junior R.P.L.,Lecture Notes in Production Engineering
2025,Design of an Intelligent Diagnosis and Self-Healing System for ZigBee Networks Based on Machine Learning,Li T.,2025 5th International Conference on Electronics Circuits and Information Engineering Ecie 2025
2025,Structural Testing of a RRAM-based AI Accelerator Core,Serlis E.A.,Proceedings of the European Test Workshop


### Apenas na query 2 (114)

year,title,authors,source
2025,Mixed-precision Neural Networks on RISC-V Cores: ISA extensions for Multi-Pumped Soft SIMD Operations,Armeniakos G.,IEEE ACM International Conference on Computer Aided Design Digest of Technical Papers Iccad
2025,Comprehensive Review on Integrating Battery Management System with Digital Twin for Real-Time EV Battery State Estimation,Patil K.,2025 1st International Conference on Aiml Applications for Engineering and Technology Icaet 2025
2025,Model of an Open-Source MicroPython Library for GSM NB-IoT,Lupandin A.,Sensors
2025,Impact of Mechanical and Architectural Signals in the Tumor Microenvironment on Melanoma,Khan Z.M.,Advanced Healthcare Materials
2025,TMI: Two-dimensional maintainability index for automotive software maintainability measurement,Xie J.,Journal of Systems Architecture
2025,AHA: Design and Evaluation of Compute-Intensive Hardware Accelerators for AMD-Xilinx Zynq SoCs Using HLS IP Flow,Berrazueta-Mena D.,Computers
2025,Focal High-Grade Areas with a Tumor-in-Tumor Pattern: Another Feature of Pediatric DICER1-Associated Thyroid Carcinoma?,Schiavo Lena M.,Endocrine Pathology
2025,Control Logic Synthesis: Drawing the Rest of the OWL,Sisco Z.D.,International Conference on Architectural Support for Programming Languages and Operating Systems ASPLOS
2025,Securing the Future: Navigating Challenges and Innovations in IoT Security,Akpabio E.,Smart Innovation Systems and Technologies
2025,Verification of AMBA APB Bus Protocol Using UVM,Saravanan S.,Proceedings 3rd International Conference on Artificial Intelligence and Machine Learning Applications Healthcare and Internet of Things Aimla 2025
